# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We will investigate which record sets (i.e., tables or files) are available in the package and print their structure. All objects will be referenced by their `@id` for unambiguous referencing.

In [ ]:
# List all record sets, their @id, and fields
record_sets = dataset.list_record_sets()

print(f'Total record sets in the dataset: {len(record_sets)}\n')
for rec in record_sets:
    print(f"Record set @id: {rec['@id']}")
    print(f"  Title: {rec.get('name', '')}")
    print(f"  Description: {rec.get('description', '')}")
    fields = dataset.list_fields(rec['@id'])
    print(f"  Fields ({len(fields)}):")
    for f in fields:
        print(f"    - @id: {f['@id']}, name: {f.get('name', '')}, dataType: {f.get('dataType', '')}")
    print('-'*60)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

We'll use the `@id` of the main record set (typically "cr:RecordSet-1" for the primary data table in mlcroissant datasets) and dynamically load all available record sets into a dictionary of DataFrames.

In [ ]:
# List all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.list_record_sets()]
print(f"Available record set @ids: {record_set_ids}\n")

dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f" - Loaded {len(dataframes[record_set_id])} records.")

# Pick the first record set for demonstration
main_record_set_id = record_set_ids[0]
print(f"\nFields/Columns in {main_record_set_id}: {dataframes[main_record_set_id].columns.tolist()}")
display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We will select a numeric field using its `@id`, filter records, normalize values, and perform grouping if relevant fields are present.

*Note:* Adjust the `numeric_field_id` and `group_field_id` as needed after listing fields above.

In [ ]:
# Choose a numeric field from the main record set for demonstration
# For instance, let's use 'cr:Field-coverage_percent' if available (change accordingly based on field listing)
main_df = dataframes[main_record_set_id]

example_numeric_field_id = None
possible_numeric_fields = ['coverage_percent', 'MW', 'pI', 'Cr_Field-coverage_percent', 'cr:Field-coverage_percent', 'cr:Field-MW', 'cr:Field-pI']
# Try to auto-detect a numeric field
for col in main_df.columns:
    if any(x.lower() in col.lower() for x in possible_numeric_fields):
        example_numeric_field_id = col
        break

if example_numeric_field_id is None:
    print("No obvious numeric field found in columns: please pick from above.")
else:
    print(f"Numeric field for EDA: {example_numeric_field_id}\n")
    # Ensure the column is numeric
    col_numeric = pd.to_numeric(main_df[example_numeric_field_id], errors='coerce')
    threshold = col_numeric.mean() if ~pd.isna(col_numeric.mean()) else 0
    filtered_df = main_df[col_numeric > threshold].copy()
    print(f"Filtered records with {example_numeric_field_id} > mean ({threshold:.2f}): {len(filtered_df)} records")

    filtered_df[f"{example_numeric_field_id}_normalized"] = (col_numeric - col_numeric.mean())/col_numeric.std()
    display(filtered_df[[example_numeric_field_id, f"{example_numeric_field_id}_normalized"]].head())

    # Try grouping by a main categorical field if present
    # Example: group by 'cr:Field-description' or similar
    group_field_id = None
    possible_group_fields = ['description', 'cr:Field-description', 'accession', 'cr:Field-accession']
    for col in main_df.columns:
        if any(x.lower() in col.lower() for x in possible_group_fields):
            group_field_id = col
            break
    if group_field_id is not None:
        grouped = (
            filtered_df.groupby(group_field_id)[example_numeric_field_id]
            .mean()
            .sort_values(ascending=False)
        )
        print(f"Grouped mean of {example_numeric_field_id} by {group_field_id} (top 5):")
        display(grouped.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the numeric field
if example_numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(pd.to_numeric(main_df[example_numeric_field_id], errors='coerce').dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {example_numeric_field_id}")
    plt.xlabel(example_numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Scatter plot if both numeric and some grouping field
    if group_field_id is not None:
        # For illustration, plot mean per group
        grouped_df = (
            main_df.groupby(group_field_id)[example_numeric_field_id]
            .mean()
            .sort_values(ascending=False)
        )
        plt.figure(figsize=(10, 4))
        grouped_df.plot(kind='bar')
        plt.title(f"Mean {example_numeric_field_id} by {group_field_id}")
        plt.ylabel(f"Mean {example_numeric_field_id}")
        plt.xlabel(group_field_id)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook has demonstrated how to use the `mlcroissant` library to explore the FAIR² mass spectrometry dataset, including loading metadata and records by `@id`, identifying record sets and fields, extracting and analyzing data, filtering and normalizing numeric values, grouping by key attributes, and visualizing important distributions.

Continue analysis by applying domain-specific processing or advanced modeling as needed.